# A-001 Simple RAG with LlamaIndex on Databricks

Goal: Answer questions from the 9 A-001 PDF documents stored in:

`/Workspace/Nuclear_Enterprise_360/A001 Documents`

Flow:

**PDFs → chunks → embeddings → vector index → top-3 retrieval → LLM → grounded answer**


In [ ]:
%pip install -q -U \
    llama-index \
    llama-index-readers-file \
    llama-index-llms-databricks \
    llama-index-embeddings-databricks \
    pypdf

dbutils.library.restartPython()


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

DOCUMENT_FOLDER = "/Workspace/Nuclear_Enterprise_360/A001 Documents"

CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-qwen3-embedding-0-6b"

CHUNK_SIZE = 400
CHUNK_OVERLAP = 100
TOP_K = 3


In [ ]:
# Check the files
import os

files = os.listdir(DOCUMENT_FOLDER)

print("Files found:")
for i, file in enumerate(files, 1):
    print(f"{i}. {file}")

print("\nTotal files:", len(files))


In [ ]:
# Configure Databricks LLM + embedding model
from llama_index.core import Settings
from llama_index.llms.databricks import Databricks
from llama_index.embeddings.databricks import DatabricksEmbedding

API_ROOT = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiUrl()
    .get()
)

API_TOKEN = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

SERVING_ENDPOINT = f"{API_ROOT}/serving-endpoints"

llm = Databricks(
    model=CHAT_MODEL,
    api_key=API_TOKEN,
    api_base=SERVING_ENDPOINT,
)

embed_model = DatabricksEmbedding(
    model=EMBED_MODEL,
    api_key=API_TOKEN,
    endpoint=SERVING_ENDPOINT,
)

Settings.llm = llm
Settings.embed_model = embed_model

print("Models configured successfully.")


In [ ]:
# Load only PDF documents
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_dir=DOCUMENT_FOLDER,
    recursive=True,
    required_exts=[".pdf"]
).load_data()

print("Documents loaded:", len(documents))


In [ ]:
# Chunk the documents
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

Settings.text_splitter = splitter

print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)


In [ ]:
# Build the vector index
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(
    documents,
    transformations=[splitter],
    show_progress=True
)

print("RAG index created.")


In [ ]:
# Create query engine
query_engine = index.as_query_engine(
    similarity_top_k=TOP_K
)

print("Query engine ready.")


In [ ]:
# Ask a first question
question = "What is asset A-001?"

response = query_engine.query(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(response)


In [ ]:
# Try another question
question = "What are the main systems or components associated with A-001?"

response = query_engine.query(question)

print(response)


In [ ]:
# Show retrieved sources
question = "What are the main issues or risks associated with A-001?"

response = query_engine.query(question)

print("=" * 80)
print("ANSWER")
print("=" * 80)
print(response)

print("\n" + "=" * 80)
print("SOURCES RETRIEVED")
print("=" * 80)

for i, node in enumerate(response.source_nodes, 1):
    file_name = node.node.metadata.get("file_name", "Unknown file")

    print(f"\nSOURCE {i}")
    print("File:", file_name)
    print("Similarity score:", round(node.score, 4) if node.score else "N/A")

    print("\nRetrieved text:")
    print(node.node.text[:700])

    print("-" * 80)
